# Convert between time zones

Three scenarios come up repeatedly: displaying a UTC timestamp in a user's local zone, converting a local time someone typed into UTC for storage, and comparing times across zones. This recipe walks through each.

## Scenario 1: UTC to local for display

You have a UTC timestamp (from a database, log, API) and want to show it in London time — or in the user's local zone, whatever that is.

In [ ]:
from datetime import datetime
from zoneinfo import ZoneInfo

utc = ZoneInfo("UTC")
london = ZoneInfo("Europe/London")

stored = datetime(2026, 4, 21, 14, 30, tzinfo=utc)
display = stored.astimezone(london)
print(display)          # 2026-04-21 15:30:00+01:00 (BST in April)
print(display.strftime("%d %B %Y, %H:%M %Z"))

`%Z` in the format string prints the zone name (`BST`, `GMT`, etc.). Useful for UIs so users know which zone they're looking at.

## Scenario 2: local to UTC for storage

A user types `"2026-09-15 15:00"` into a form, indicating London time. You want to store it in UTC in the database.

In [ ]:
entered = datetime(2026, 9, 15, 15, 0, tzinfo=london)    # attach the user's zone
for_storage = entered.astimezone(utc)
print("London:", entered)
print("UTC:   ", for_storage)

The transformation is one-way for storage: once you have the UTC instant you can always convert back. The opposite — storing the local time without a zone — loses information at DST transitions, so always convert to aware UTC before persisting.

## Scenario 3: comparing times across zones

Is a 15:00 London meeting after or before a 09:00 New York meeting? Both are aware, so Python can compare them directly — but you might want to normalise for sanity.

In [ ]:
new_york = ZoneInfo("America/New_York")

london_mtg = datetime(2026, 9, 15, 15, 0, tzinfo=london)
ny_mtg = datetime(2026, 9, 15, 9, 0, tzinfo=new_york)

print(london_mtg > ny_mtg)                    # comparison works on aware datetimes
print(london_mtg.astimezone(utc))             # 14:00 UTC
print(ny_mtg.astimezone(utc))                 # 13:00 UTC
# So the NY meeting starts one hour earlier than the London one.

## Scenario 4: "the same wall-clock time in every zone"

Occasionally you want the opposite — "send the reminder at 09:00 **local time** for each user". That's not a single instant; it's a different instant per zone. Build it per-user by attaching their zone:

In [ ]:
users = {
    "alice":   london,
    "bob":     new_york,
    "carmen":  ZoneInfo("Asia/Tokyo"),
}

for name, zone in users.items():
    local_9am = datetime(2026, 9, 15, 9, 0, tzinfo=zone)
    print(f"{name:8} local 09:00 = {local_9am.astimezone(utc)} UTC")

Three different UTC instants. Schedule the reminder at each user's UTC equivalent.

## DST watch-outs

On the day the clocks change, "09:00 local" shifts by an hour in UTC relative to the day before. `zoneinfo` handles this correctly — the key is to attach the zone *before* any arithmetic, not after.

Don't do `datetime(2026, 3, 29, 9, 0) - timedelta(days=1)` then attach the zone — you'll get the wrong thing. Attach the zone first, then do arithmetic on aware values.

In [ ]:
from datetime import timedelta

# Correct: zone first, then arithmetic
mtg = datetime(2026, 3, 29, 9, 0, tzinfo=london)
print("Meeting:      ", mtg.astimezone(utc))       # 08:00 UTC (BST)
print("Same mtg -1d: ", (mtg - timedelta(days=1)).astimezone(utc))   # 09:00 UTC (GMT)
# The UTC offset is different because we crossed the spring-forward boundary.

Elapsed real time is still 24 hours — `mtg - (mtg - timedelta(days=1))` is `timedelta(days=1)` — but the UTC representation shifts because London's offset from UTC did.

## Tips

- **Store everything in UTC.** Convert on display, not on storage. See the [UTC everywhere essay](../concepts/utc-everywhere.md).
- **Attach zones before arithmetic.** Doing arithmetic on naive datetimes and attaching zones afterwards loses DST information.
- **Prefer IANA names** (`Europe/London`) over offsets (`+01:00`) — offsets don't know about DST transitions.
- **`%Z` in `strftime`** prints the zone abbreviation for display.